# Resume Keyword Analyzer

This notebook shows the main workflow for the CSEN/CSCI 185 final project. It loads the job postings dataset, filters for a selected job title, and runs keyword frequency, TF-IDF, association rule mining, and visualization steps.

## Imports

The notebook uses the helper functions already defined in the `src` folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_jobs_csv, inspect_dataset, filter_by_job_title
from src.keyword_frequency import compute_keyword_frequency
from src.tfidf_analysis import run_tfidf_analysis
from src.association_rules import extract_skill_transactions, mine_skill_rules, format_rules
import src.visualizations as visualizations
visualizations.OUTPUTS_DIR = str(PROJECT_ROOT / "outputs")
from src.visualizations import save_all_charts

## Load Dataset

The dataset should be placed locally at `data/jobs.csv`.

In [ ]:
jobs_df = load_jobs_csv(PROJECT_ROOT / "data" / "jobs.csv")
inspect_dataset(jobs_df)

## Filter By Job Title

For this example, the notebook analyzes data analyst postings.

In [ ]:
selected_job_title = "data analyst"
filtered_df = filter_by_job_title(jobs_df, selected_job_title)

print(f"Selected job title: {selected_job_title}")
print(f"Matching postings: {len(filtered_df)}")
filtered_df[["title", "company_name", "location"]].head()

## Description Check

The analysis functions handle preprocessing internally. This cell checks that there are descriptions available for the selected role.

In [ ]:
descriptions = filtered_df["description"].dropna()
descriptions = descriptions[descriptions.str.strip() != ""]

print(f"Descriptions available: {len(descriptions)}")

if len(descriptions) > 0:
    print(descriptions.iloc[0][:800])
else:
    print("No descriptions available for this selected role.")

## Keyword Frequency

This section finds the most common keywords and skills in the filtered job descriptions.

In [ ]:
top_keywords_df, top_skills_df = compute_keyword_frequency(filtered_df, top_n=20)

top_keywords_df.head(10)

In [ ]:
top_skills_df.head(10)

## TF-IDF Keywords

TF-IDF helps identify terms that are important for the selected role.

In [ ]:
top_tfidf_df, tfidf_vectorizer, tfidf_matrix = run_tfidf_analysis(filtered_df, top_n=20)

top_tfidf_df.head(10)

## Association Rules

Each job posting is treated like a transaction, and each detected skill is treated like an item.

In [ ]:
transactions = extract_skill_transactions(filtered_df["description"])
print(f"Skill transactions: {len(transactions)}")

rules_df = mine_skill_rules(transactions, min_support=0.05, min_confidence=0.3)
formatted_rules_df = format_rules(rules_df, top_n=10)

formatted_rules_df

## Visualizations

Charts are saved in the `outputs` folder. The word cloud step is optional and will be skipped if the package is unavailable.

In [ ]:
saved_charts = save_all_charts(top_keywords_df, top_skills_df, top_tfidf_df, selected_job_title, top_n=15)
saved_charts

## Summary

This notebook demonstrates the main project workflow for one selected job title. The same steps can be repeated for other roles such as software engineer or data scientist.